# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# Some imports for handling images ->
import os,json
import requests
import urllib
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import edge_tts
import tempfile
import asyncio

In [ ]:
load_dotenv(override=True)
groq_api_key=os.getenv('GROQ_API_KEY')
if groq_api_key:
        print(f"Groq api key is present and begins with the {groq_api_key[:8]}")
else:
        print(f"No API KEY found")
MODEL = "llama-3.3-70b-versatile"
openai = OpenAI()
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

In [ ]:
DB = os.path.join(os.getcwd(), "answers.db")
print("DB path:", DB) 

In [ ]:
import os, sqlite3

# Use CWD directly — we confirmed it's already the right folder
DB = os.path.join(os.getcwd(), "answers.db")
print("DB path:", DB)

# Create and seed
with sqlite3.connect(DB) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS questions (
            question TEXT PRIMARY KEY,
            answer   TEXT NOT NULL
        )
    """)
    conn.executemany(
        "INSERT OR IGNORE INTO questions (question, answer) VALUES (?, ?)",
        [
            ("what is react",  "React is a JavaScript library for building UIs."),
            ("what is golang", "Go is a compiled language designed at Google."),
            ("what is docker", "Docker packages apps into containers."),
        ]
    )
    conn.commit()
    rows = conn.execute("SELECT * FROM questions").fetchall()
    print(f"{len(rows)} rows ready:")
    for r in rows: print(" ", r)

In [ ]:
def get_answer(question: str) -> str:
    DB = os.path.join(os.getcwd(), "answers.db")
    with sqlite3.connect(DB) as conn:
        result = conn.execute(
            "SELECT answer FROM questions WHERE question = ?",
            (question.lower().strip(),)
        ).fetchone()
    return f"Answer: {result[0]}" if result else "No answer found."

In [ ]:

answer_function = {
    "name": "get_answer",
    "description": "Look up answer to a technical question from the database. Call this for questions about React, Golang, Docker, Node.js, DevOps, AI/ML.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The technical question to look up, e.g. 'what is react'"
            }
        },
        "required": ["question"]
    }
}

tools = [{"type": "function", "function": answer_function}]

In [ ]:
# Image generation ->
def artist(topic: str) -> Image.Image:
        prompt = (
                f"{topic} - vibrant pop-art style, highly detailed, "
                "showing key features of the questions and atmosphere"
        )
        encoded = urllib.parse.quote(prompt)
        response = requests.get(
                f"https://image.pollinations.ai/prompt/{encoded}"
                f"?width=1024&height=1024&model=flux",
                timeout=60,
        )
        response.raise_for_status()
        return Image.open(BytesIO(response.content))

In [ ]:
# Text to Speech ->
def talker(message:str) -> str:
        """Convert *message* to the speech and return the temp .mp3 file path."""
        async def _run():
                communicate = edge_tts.Communicate(message,voice="en-US-GuyNeural")
                with tempfile.NamedTemporaryFile(delete=False,suffix=".mp3") as f:
                        await communicate.save(f.name)
                        return f.name
        return asyncio.run(_run())

In [ ]:
SYSTEM_PROMPT = """
You are a FAANG-level senior engineer.

You specialize in:
- React
- Next.js
- Node.js
- Golang
- DevOps
- AI/ML
- System Design

Rules:
- Explain simply
- Give production examples
- Use clean code
- Use bullet points
- Help beginners understand deeply
"""

In [ ]:
# Tool Call Handler =>
TOOL_MAP = {"get_answer":get_answer}

def handle_tool_calls(tool_calls) -> list[dict]:
    results = []
    for tc in tool_calls:
        fn_name = tc.function.name
        try:
            fn_args = json.loads(tc.function.arguments)
        except json.JSONDecodeError:
            fn_args = {"question": tc.function.arguments}  # fallback
        
        print(f"[tool] calling {fn_name}({fn_args})")
        output = TOOL_MAP.get(fn_name, lambda **k: "Tool not found")(**fn_args)
        results.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": str(output),
        })
    return results

In [ ]:
def chat(user_message: str, history: list[dict]):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history
    messages.append({"role": "user", "content": user_message})
    db_keywords = ["what is", "explain", "define", "tell me about"]
    use_db = any(kw in user_message.lower() for kw in db_keywords)

    if use_db:
        topic = user_message.lower()
        for kw in db_keywords:
            topic = topic.replace(kw, "").strip()
        
        db_result = get_answer(topic)  # check DB first
        
        if "No answer found" not in db_result:
            messages[-1]["content"] = (
                f"{user_message}\n\n"
                f"[Context from database: {db_result}] "
                f"Use this to answer, then expand further."
            )

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=1024,
    )
    reply = response.choices[0].message.content

    new_history = history + [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": reply},
    ]

    audio_path = talker(reply)

    image = None
    image_keywords = ["image", "picture", "photo", "show", "visualize", "draw"]
    if any(kw in user_message.lower() for kw in image_keywords):
        image = artist(user_message)

    return new_history, audio_path, image

In [ ]:
# Gradio UI ->
def respond(user_message: str, history: list):
    """Wrapper called by Gradio; threads state through the chat function."""
    if not user_message.strip():
        return history, history, None, None

    new_history, audio_path, image = chat(user_message, history)
    return new_history, new_history, audio_path, image

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🤖 AI Technical Assistant")

    chatbot = gr.Chatbot(height=500) 

    with gr.Row():
        msg = gr.Textbox(
            label="Ask Anything…",
            placeholder="e.g. What is React?  /  Show me an image of Tokyo",
            scale=3,
        )
        send_btn = gr.Button("Send", variant="primary", scale=1)

    with gr.Row():
        image_output = gr.Image(label="Generated Image")
        audio_output = gr.Audio(label="Voice Response", autoplay=True)

    state = gr.State([])   

    for trigger in (msg.submit, send_btn.click):
        trigger(
            fn=respond,
            inputs=[msg, state],
            outputs=[chatbot, state, audio_output, image_output],
        ).then(lambda: "", outputs=msg)   

demo.launch(inbrowser=True, auth=("adi", "123456"))